# 01 Data Inspection and Preprocessing

This notebook keeps the modeling workflow separate from inference. It loads the raw CSV, inspects the columns, selects `depression_label` as the target, splits the data, encodes categorical features, scales numeric features, applies SMOTE only to the training data, and saves the balanced training set.

In [ ]:
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import SMOTE

In [ ]:
from google.colab import files
print('Select dataset (CSV) to upload when prompted.')
uploaded = files.upload()
if uploaded:
    fname = list(uploaded.keys())[0]
    df = pd.read_csv(fname)
print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
display(df.head())
print(df.info())

In [ ]:
target_col = 'depression_label'
X = df.drop(target_col, axis=1)
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train counts:')
print(y_train.value_counts())
print('y_test counts:')
print(y_test.value_counts())

In [ ]:
# Separate feature columns by type so we can preprocess them correctly.
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object']).columns.tolist()
print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)

# Standardize numeric columns and one-hot encode categorical columns.
# This converts everything into numeric form before SMOTE runs.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ]
)
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)
print('Encoded train shape:', X_train_prepared.shape)
print('Encoded test shape :', X_test_prepared.shape)

# SMOTE balances only the training target labels.
# It does not touch the test set.
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_prepared, y_train)
print('Before SMOTE:')
print(y_train.value_counts())
print('After SMOTE:')
print(y_train_res.value_counts())
print('Prepared train shape:', X_train_prepared.shape)
print('Balanced train shape:', X_train_res.shape)

In [ ]:
from google.colab import files

feature_names = preprocessor.get_feature_names_out()
train_bal = pd.DataFrame(X_train_res, columns=feature_names)
train_bal[target_col] = y_train_res
output_file = 'train_balanced.csv'
train_bal.to_csv(output_file, index=False)
files.download(output_file)
print(f'Created and downloaded {output_file}')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Train a simple baseline model on the balanced training data.
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_res, y_train_res)

# Evaluate only on the untouched test split.
y_pred = model.predict(X_test_prepared)

print('Accuracy :', accuracy_score(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred, zero_division=0))
print('Recall   :', recall_score(y_test, y_pred, zero_division=0))
print('F1 Score :', f1_score(y_test, y_pred, zero_division=0))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('Classification Report:')
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Second model: a tree-based baseline for comparison.
second_model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
second_model.fit(X_train_res, y_train_res)
second_pred = second_model.predict(X_test_prepared)

comparison = pd.DataFrame({
    'model': ['Logistic Regression', 'Random Forest'],
    'accuracy': [accuracy_score(y_test, y_pred), accuracy_score(y_test, second_pred)],
    'precision': [precision_score(y_test, y_pred, zero_division=0), precision_score(y_test, second_pred, zero_division=0)],
    'recall': [recall_score(y_test, y_pred, zero_division=0), recall_score(y_test, second_pred, zero_division=0)],
    'f1_score': [f1_score(y_test, y_pred, zero_division=0), f1_score(y_test, second_pred, zero_division=0)],
})

print('Second model confusion matrix:')
print(confusion_matrix(y_test, second_pred))
print('Second model classification report:')
print(classification_report(y_test, second_pred, zero_division=0))
print('Model comparison:')
display(comparison)

In [ ]:
# Stable model comparison with stratified k-fold cross-validation
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

# Build per-fold pipelines to avoid data leakage during validation.
cv_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ]
)

cv_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
}

cv_rows = []
for name, clf in cv_models.items():
    pipe = ImbPipeline(steps=[
        ('preprocess', cv_preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('model', clf),
    ])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_rows.append({
        'model': name,
        'cv_accuracy_mean': scores['test_accuracy'].mean(),
        'cv_accuracy_std': scores['test_accuracy'].std(),
        'cv_precision_mean': scores['test_precision'].mean(),
        'cv_recall_mean': scores['test_recall'].mean(),
        'cv_f1_mean': scores['test_f1'].mean(),
    })

cv_comparison = pd.DataFrame(cv_rows).sort_values('cv_f1_mean', ascending=False)
print('5-fold CV comparison (mean across folds):')
display(cv_comparison)


## Decision Summary (From 5-Fold CV)

- Logistic Regression is preferred when the project goal is to identify as many depression-positive cases as possible (higher recall and slightly better F1).
- Random Forest is preferred only if minimizing false positives is the top priority (higher precision).
- Accuracy is high for both models, but with class imbalance, recall and F1 are more reliable selection metrics.
- For this mental-health screening objective, choose **Logistic Regression** as the current baseline model.

Yes, you can do the same preprocessing for other targets too, if those columns are valid prediction targets.

The pattern is:

choose a target column
treat all other relevant columns as features
split into train/test
encode and scale the features
apply SMOTE only if that target is imbalanced
train a model for that target
So if your dataset has another label you want to predict, you can repeat the same workflow for it.

A few important conditions:

The target should be a classification label if you want to use SMOTE.
If the target is continuous, like a score or measurement, then this becomes a regression problem, and SMOTE is not the right step.
If a column is not a prediction target but just another input feature, then it stays in X, not in y.
So the answer is yes, but only for columns you actually want to predict. If you want, I can show you how to turn this notebook into a reusable function so you can run it for different target columns.

In [ ]:
# Data cleaning helper
def clean_dataset(df_in):
    """Basic cleaning: drop duplicates, report and fill missing values.
    Numeric columns: fill with median. Categorical: fill with mode.
    Returns a cleaned copy of the dataframe."""
    df_clean = df_in.copy()
    # drop exact duplicates
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    after = len(df_clean)
    print(f'Dropped {before-after} duplicate rows')

    print('\nMissing values per column:')
    print(df_clean.isna().sum())

    num_cols = df_clean.select_dtypes(include=['number']).columns
    cat_cols = df_clean.select_dtypes(include=['object', 'category']).columns

    for c in num_cols:
        if df_clean[c].isna().any():
            med = df_clean[c].median()
            df_clean[c] = df_clean[c].fillna(med)
            print(f'Filled NA in numeric {c} with median={med}')

    for c in cat_cols:
        if df_clean[c].isna().any():
            mode = df_clean[c].mode()
            fill = mode.iloc[0] if not mode.empty else ''
            df_clean[c] = df_clean[c].fillna(fill)
            print(f"Filled NA in categorical {c} with mode='{fill}'")

    return df_clean

# Example usage (uncomment to run):
# df = clean_dataset(df)
